# 🧠 EMG Biosignal - CTS Classification Tutorial

Sistem klasifikasi tingkat keparahan **Carpal Tunnel Syndrome (CTS)** menggunakan sinyal **EMG (Electromyography)** dan **Deep Learning**.

## 📊 Pipeline Keseluruhan:
```
Raw EMG Signal → Preprocessing → Feature Extraction → Model Training → Evaluation
```

---

## 1️⃣ CELL 1: Import Libraries

Semua library yang dibutuhkan diimport di satu cell pertama.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Import dari modul custom project
from data_preprocessing import EMGPreprocessor, preprocess_batch_signals_to_spectrograms
from feature_extraction import SpectrogramExtractor, extract_batch_spectrograms
from models import ModelBuilder
from train import DataSplitter, ModelTrainer
from evaluate import ModelEvaluator

print("✓ Semua library berhasil diimport!")

## 2️⃣ CELL 2: Load Configuration

Baca file `config.json` untuk mendapatkan setting project.

In [ ]:
# Load config
with open('config.json', 'r') as f:
    config = json.load(f)

# Ekstrak parameter penting
data_base_dir = config['data']['base_directory']
sampling_rate = config['signal_processing']['sampling_rate']
classes = config['data']['classes']
class_mapping = config['data']['class_mapping']

print(f"✓ Config berhasil dimuat")
print(f"  📁 Data Directory: {data_base_dir}")
print(f"  🎵 Sampling Rate: {sampling_rate} Hz")
print(f"  📋 Kelas: {classes}")
print(f"  🔢 Class Mapping: {class_mapping}")

## 3️⃣ CELL 3: Load & Explore EMG Data

Memuat file EMG dari folder struktur dan melihat statistik data.

In [ ]:
# Scan EMG files
def load_emg_data(data_dir):
    """Load semua file EMG dari struktur folder"""
    file_paths = {}
    for class_name in classes:
        class_dir = os.path.join(data_dir, class_name)
        if os.path.exists(class_dir):
            files = [f for f in os.listdir(class_dir) if f.endswith(('.txt', '.csv'))]
            file_paths[class_name] = [os.path.join(class_dir, f) for f in files]
        else:
            file_paths[class_name] = []
    return file_paths

# Ganti dengan path yang sesuai atau gunakan Motorik/Sensorik/Full_Data
DATA_FOLDER = os.path.join(data_base_dir, 'Motorik')  # atau 'Sensorik', 'Full_Data'

# Load
file_paths = load_emg_data(DATA_FOLDER)

# Tampilkan statistik
print("📊 Data Statistics:")
for class_name, files in file_paths.items():
    print(f"  {class_name}: {len(files)} files")

total_files = sum(len(files) for files in file_paths.values())
print(f"\n✓ Total files: {total_files}")

## 4️⃣ CELL 4: Load & Visualize Single EMG Signal

Lihat bentuk sinyal EMG mentah sebelum diproses.

In [ ]:
# Load sample signal
if file_paths['non_cts']:
    sample_file = file_paths['non_cts'][0]
    sample_signal = np.loadtxt(sample_file)
    
    # Reshape jika perlu
    if len(sample_signal.shape) == 1:
        sample_signal = sample_signal.reshape(-1, 1)
    
    print(f"📂 Sample file: {os.path.basename(sample_file)}")
    print(f"📏 Shape: {sample_signal.shape}")
    print(f"📊 Mean: {sample_signal.mean():.4f}")
    print(f"📊 Std: {sample_signal.std():.4f}")
    print(f"📊 Min: {sample_signal.min():.4f}")
    print(f"📊 Max: {sample_signal.max():.4f}")
    
    # Visualisasi
    fig, ax = plt.subplots(figsize=(14, 4))
    time_axis = np.arange(sample_signal.shape[0]) / sampling_rate
    ax.plot(time_axis, sample_signal[:, 0], linewidth=0.5)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Amplitude (μV)')
    ax.set_title(f'Raw EMG Signal - non_cts')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("❌ Tidak ada file non_cts. Periksa path data.")

## 5️⃣ CELL 5: Signal Preprocessing

Membersihkan sinyal dengan filtering dan normalisasi.

In [ ]:
# Initialize preprocessor
preprocessor = EMGPreprocessor(
    sampling_rate=sampling_rate,
    lowcut=20,
    highcut=500,
    order=4
)

# Preprocess sample signal
cleaned_signal = preprocessor.apply_bandpass_filter(sample_signal[:, 0])
normalized_signal = preprocessor.normalize_signal(cleaned_signal)

print("✓ Preprocessing selesai!")
print(f"  Cleaned signal shape: {cleaned_signal.shape}")
print(f"  Normalized signal - Mean: {normalized_signal.mean():.4f}, Std: {normalized_signal.std():.4f}")

# Visualisasi perbandingan
fig, axes = plt.subplots(3, 1, figsize=(14, 8))
time_axis = np.arange(len(sample_signal[:, 0])) / sampling_rate

axes[0].plot(time_axis, sample_signal[:, 0], color='red', linewidth=0.5)
axes[0].set_title('Raw Signal')
axes[0].set_ylabel('Amplitude')
axes[0].grid(True, alpha=0.3)

axes[1].plot(time_axis, cleaned_signal, color='green', linewidth=0.5)
axes[1].set_title('After Bandpass Filter')
axes[1].set_ylabel('Amplitude')
axes[1].grid(True, alpha=0.3)

axes[2].plot(time_axis, normalized_signal, color='blue', linewidth=0.5)
axes[2].set_title('After Normalization')
axes[2].set_ylabel('Amplitude')
axes[2].set_xlabel('Time (s)')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6️⃣ CELL 6: Feature Extraction (Spectrogram)

Konversi sinyal ke domain frekuensi menggunakan STFT dan buat spectrogram.

In [ ]:
from scipy import signal as scipy_signal

# Compute STFT
frequencies, times, spectrogram_data = scipy_signal.spectrogram(
    normalized_signal,
    fs=sampling_rate,
    window='hann',
    nperseg=512,
    noverlap=256
)

# Convert to dB scale
spectrogram_db = 10 * np.log10(spectrogram_data + 1e-10)

print(f"✓ Spectrogram generated!")
print(f"  Shape: {spectrogram_db.shape}")
print(f"  Frequencies: {frequencies.min():.2f} - {frequencies.max():.2f} Hz")
print(f"  Time duration: {times.min():.2f} - {times.max():.2f} s")

# Visualisasi
fig, ax = plt.subplots(figsize=(12, 6))
im = ax.pcolormesh(times, frequencies, spectrogram_db, shading='auto', cmap='viridis')
ax.set_ylabel('Frequency (Hz)')
ax.set_xlabel('Time (s)')
ax.set_title('EMG Spectrogram (dB)')
ax.set_ylim([0, 500])  # Focus on relevant frequency range
cbar = plt.colorbar(im, ax=ax, label='Power (dB)')
plt.tight_layout()
plt.show()

## 7️⃣ CELL 7: Model Architecture

Membangun neural network CNN untuk klasifikasi.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

def build_cnn_model(input_shape, num_classes):
    """Build CNN model untuk EMG classification"""
    model = keras.Sequential([
        # Block 1
        layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=input_shape),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.3),
        
        # Block 2
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.3),
        
        # Block 3
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.4),
        
        # Flatten & Dense
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])
    
    return model

# Build model
input_shape = (128, 128, 1)  # Spectrogram height, width, channels
num_classes = len(classes)

model = build_cnn_model(input_shape, num_classes)

# Compile
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("✓ Model built successfully!")
model.summary()

## 8️⃣ CELL 8: Data Preparation untuk Training

Prepare dataset dan split ke train/val/test.

In [ ]:
from sklearn.model_selection import train_test_split

# Dummy data untuk contoh (dalam praktik, load dari file sebenarnya)
# Generate random spectrograms
X_dummy = np.random.randn(100, 128, 128, 1)  # 100 samples
y_dummy = np.random.randint(0, num_classes, 100)
y_dummy_onehot = keras.utils.to_categorical(y_dummy, num_classes)

# Split data
X_train, X_temp, y_train, y_temp = train_test_split(
    X_dummy, y_dummy_onehot, test_size=0.3, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

print(f"✓ Data split completed!")
print(f"  Train: {X_train.shape[0]} samples")
print(f"  Validation: {X_val.shape[0]} samples")
print(f"  Test: {X_test.shape[0]} samples")
print(f"\n  Total: {X_train.shape[0] + X_val.shape[0] + X_test.shape[0]} samples")

## 9️⃣ CELL 9: Model Training

Latih model dengan data training.

In [ ]:
# Callbacks
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

# Train model
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

print("✓ Training completed!")

## 🔟 CELL 10: Training History & Metrics

Visualisasi hasil training dan evaluasi model.

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Accuracy
axes[0].plot(history.history['accuracy'], label='Train Accuracy')
axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Model Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss
axes[1].plot(history.history['loss'], label='Train Loss')
axes[1].plot(history.history['val_loss'], label='Validation Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('Model Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Evaluate on test set
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"\n📊 Test Results:")
print(f"  Loss: {test_loss:.4f}")
print(f"  Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")

## 1️⃣1️⃣ CELL 11: Confusion Matrix & Classification Report

Analisis detail performa model per kelas.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

# Predictions
y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)
y_test_labels = np.argmax(y_test, axis=1)

# Confusion matrix
cm = confusion_matrix(y_test_labels, y_pred)

# Plot confusion matrix
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=classes, yticklabels=classes, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('Confusion Matrix')
plt.tight_layout()
plt.show()

# Classification report
print("\n📋 Classification Report:")
print(classification_report(y_test_labels, y_pred, target_names=classes))

## 1️⃣2️⃣ CELL 12: Prediction pada Data Baru

Gunakan model untuk memprediksi kelas CTS dari sinyal baru.

In [ ]:
# Buat sample spectrogram baru
new_sample = np.random.randn(1, 128, 128, 1)

# Predict
predictions = model.predict(new_sample, verbose=0)
predicted_class_idx = np.argmax(predictions[0])
predicted_class = classes[predicted_class_idx]
confidence = predictions[0][predicted_class_idx] * 100

print(f"🎯 Prediction Result:")
print(f"\n  Predicted Class: {predicted_class}")
print(f"  Confidence: {confidence:.2f}%")
print(f"\n📊 All Probabilities:")
for class_name, prob in zip(classes, predictions[0]):
    print(f"  {class_name}: {prob*100:.2f}%")

## 📚 Summary & Next Steps

Notebook ini mendemonstrasikan seluruh pipeline:

1. **Load & Explore Data** - Memahami struktur sinyal EMG
2. **Preprocessing** - Filter dan normalisasi sinyal
3. **Feature Extraction** - Convert ke spectrogram
4. **Model Building** - Build CNN architecture
5. **Training** - Train dengan train/val split
6. **Evaluation** - Evaluasi dengan test set
7. **Prediction** - Gunakan model untuk prediksi baru

### ✅ Tips Penggunaan:
- Run cells **secara berurutan** dari atas ke bawah
- Modifikasi parameter (learning rate, batch size, dll) di cell masing-masing
- Gunakan `%%time` magic command untuk timing eksekusi
- Simpan model dengan `model.save('model_name.h5')`

---